# PaperMind · 论文创新点发现 Agent

## 项目简介
PaperMind 是一个面向研究生的多 Agent 协作系统：批量阅读本地 PDF 论文 → 漏斗式信息收敛 → 识别研究空白 → 生成 2-3 个可落地的创新点与完整实验方案。

本项目构建在 **HelloAgents 框架** 之上：LLM 调用走框架 `HelloAgentsLLM`，工具经桥接注册为框架 `ToolRegistry`，质量审核由框架 `SimpleAgent` 执行。

## 作者信息
- 姓名：jphJack
- GitHub：[@jphJack](https://github.com/jphJack)

## 第 1 部分：环境配置

In [ ]:
# 安装依赖（HelloAgents 框架）
!pip install -q "hello-agents[all]"

In [ ]:
# 导入 HelloAgents 框架与 PaperMind 组件
import os
import sys

sys.path.append(os.path.abspath("."))  # 使 backend 包可导入

from hello_agents import HelloAgentsLLM, ReActAgent, SimpleAgent

from backend.config import settings
from backend.llm.client import DeepSeekClient
from backend.tools.registry import build_default_registry
from backend.tools.hello_agents_bridge import build_hello_agents_registry
from backend.agents.hello_agents_runner import build_reviewer_agent, build_react_agent

print("HelloAgents 框架与 PaperMind 组件导入成功")

## 第 2 部分：HelloAgents 框架 LLM 客户端

PaperMind 的全部模型调用经框架 `HelloAgentsLLM` 发出（OpenAI 兼容适配器自动识别 DeepSeek）。

In [ ]:
# DeepSeekClient 底层即框架 HelloAgentsLLM
client = DeepSeekClient()
print("底层 LLM 类型:", type(client.llm).__name__)
print("模型:", client.model)
print("服务地址:", settings.DEEPSEEK_BASE_URL)

## 第 3 部分：工具桥接（项目工具 → 框架 ToolRegistry）

项目的 10+ Function Calling 工具经桥接成为框架原生 Tool（ToolResponse 协议 + 熔断器保护）。

In [ ]:
# 构建项目工具注册表，并桥接为框架 ToolRegistry
project_registry = build_default_registry()
print("项目工具:", project_registry.names())

ha_registry = build_hello_agents_registry(project_registry)
print("框架工具:", ha_registry.list_tools())

## 第 4 部分：框架 Agent 构建

- `SimpleAgent`：Reflexion 质量审核（纯推理）
- `ReActAgent`：携带完整工具集的推理-行动调度

In [ ]:
# 质量审核 Agent（框架 SimpleAgent）
reviewer = build_reviewer_agent(client.llm)
print("审核 Agent:", type(reviewer).__name__, "-", reviewer.name)

# GapAnalyzer 步骤的框架 ReActAgent（携带桥接工具集）
gap_agent = build_react_agent(
    name="GapAnalyzer",
    llm=client.llm,
    registry=project_registry,
    system_prompt="你是研究空白（Gap）识别专家，使用 analyze_gaps 与 hybrid_search 工具识别四类研究空白。",
    tool_names=["analyze_gaps", "hybrid_search"],
)
print("步骤 Agent:", type(gap_agent).__name__, "-", gap_agent.name)
print("已注册工具:", gap_agent.tool_registry.list_tools())

## 第 5 部分：集成自检

无需真实 API Key 的冒烟验证，确认三层集成接线正确。

In [ ]:
# 冒烟自检
from hello_agents.tools.response import ToolStatus

tool = ha_registry.get_tool("scan_folder")
schema = tool.to_openai_schema()
assert schema["function"]["name"] == "scan_folder"
assert type(client.llm).__name__ == "HelloAgentsLLM"
assert type(reviewer).__name__ == "SimpleAgent"
assert type(gap_agent).__name__ == "ReActAgent"
print("HelloAgents 三层集成自检通过：LLM 层 / 工具层 / Agent 层")

## 第 6 部分：完整流水线

配置 `DEEPSEEK_API_KEY` 后，完整七步漏斗流水线（PDF 解析 → 结构化抽取 → 向量索引 → Gap 识别 → 创新点生成 → 实验方案设计 → 整合报告）由后端 API 提供：

```bash
python run_backend.py   # http://0.0.0.0:8000
cd frontend && npm run dev   # http://localhost:5173
```

详见 [README.md](../README.md)。